# run02_v2dynamic2class_baseline（2026-10-27）
- 檔名規則：YOLOv8n_runXX_資料版本_說明_日期.ipynb；每個版本使用獨立檔案，變更記錄寫在本區，結論寫在檔尾。
- 前一版：run01_v2raw5class_baseline_20261023（5 類 raw baseline，test mAP50 0.382；person 0.897，車輛類不可用）；本檔由 run01 另存副本，run01 不再更動。
- 資料：Roboflow alert-yolo-v1 version 2（v2-raw-640-5class-fix0524），Train 499 / Valid 90 / Test 75；影像不變、無擴增。
- 變更：標籤在 Colab 合併，car / bus / motorcycle / bicycle -> dynamic_obstacle，改為 2 類（person、dynamic_obstacle）；原因是 run01 車輛類樣本太少、5 類拆分學不起來。不使用 Roboflow 點數。
- 設定：與 run01 完全相同（yolov8n.pt, imgsz=320, epochs=100, patience=20, batch=32, seed=0, deterministic, weight_decay=0.0005, 內建擴增全關），僅類別數與資料集不同，以確保可與 run01 比較。
- 用途：作為現有資料下的正式 raw baseline（老師建議 baseline 不使用 augmentation）；後續擴增實驗（run03 起）與補資料後的新 baseline 都以此對照。
- 輸出位置：Google Drive/capstone_yolo/run02_v2dynamic2class_baseline/（baseline_dyn_raw）。
- 狀態：已於 2026-10-27 執行完成（test mAP50 0.688；person 0.866、dynamic_obstacle 0.51），結論見檔尾。

In [ ]:
# [步驟1] 確認 GPU 型號（T4），並安裝 ultralytics（YOLOv8）與 roboflow 套件
!nvidia-smi -L && pip -q install ultralytics roboflow

GPU 0: Tesla T4 (UUID: GPU-545067af-0406-a678-0037-f191b20c5254)


In [ ]:
# [步驟2] 下載 Roboflow alert-yolo-v1 的 version 2（v2-raw-640-5class-fix0524，5 類，無擴增）
from google.colab import userdata; from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace('ssudscompanionrobot').project('alert-yolo-v1').version(2).download('yolov8')
print(ds.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to alert-yolo-v1-2 in yolov8:: 100%|██████████| 1332/1332 [00:00<00:00, 6701.96it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/content/alert-yolo-v1-2


In [ ]:
# [步驟3] 標籤合併：car/bus/motorcycle/bicycle -> dynamic_obstacle（id=1），person -> id=0；只改標籤檔，影像不動，輸出到 /content/ds_dyn
import yaml, shutil, glob, os
src = ds.location; dst = '/content/ds_dyn'
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
names = yaml.safe_load(open(src + '/data.yaml'))['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
new_id = {i: (0 if n == 'person' else 1) for i, n in enumerate(names)}
cnt = {}
for f in glob.glob(dst + '/*/labels/*.txt'):
    split = f.split('/')[-3]; out = []
    for line in open(f).read().splitlines():
        p = line.split()
        if len(p) < 5: continue
        p[0] = str(new_id[int(p[0])]); out.append(' '.join(p))
        cnt[(split, p[0])] = cnt.get((split, p[0]), 0) + 1
    open(f, 'w').write('\n'.join(out) + ('\n' if out else ''))
cfg = yaml.safe_load(open(dst + '/data.yaml'))
cfg.update(path=dst, train=dst + '/train/images', val=dst + '/valid/images', test=dst + '/test/images', nc=2, names=['person', 'dynamic_obstacle'])
yaml.safe_dump(cfg, open(dst + '/data.yaml', 'w'))
print(names, '->', ['person', 'dynamic_obstacle'])
print('各切分實例數 (split, 類別id): ', dict(sorted(cnt.items())))

['bicycle', 'bus', 'car', 'motorcycle', 'person'] -> ['person', 'dynamic_obstacle']
各切分實例數 (split, 類別id):  {('test', '0'): 103, ('test', '1'): 31, ('train', '0'): 508, ('train', '1'): 158, ('valid', '0'): 81, ('valid', '1'): 15}


In [ ]:
# [步驟4] 以與 run01 相同設定訓練 2 類模型（YOLOv8n、imgsz=320、100 epochs、patience=20、seed=0、擴增全關），再用 best.pt 在 test 集評估
from ultralytics import YOLO
m = YOLO('yolov8n.pt')
m.train(data=dst + '/data.yaml', imgsz=320, epochs=100, patience=20, batch=32, seed=0, deterministic=True, mosaic=0, hsv_h=0, hsv_s=0, hsv_v=0, fliplr=0, scale=0, translate=0, degrees=0, weight_decay=0.0005, project='runs', name='baseline_dyn_raw', plots=False, verbose=False)
best = YOLO(str(m.trainer.best))
r = best.val(data=dst + '/data.yaml', split='test', imgsz=320, plots=False)
print('TEST mAP50', r.box.map50, 'mAP50-95', r.box.map, 'P', r.box.mp, 'R', r.box.mr)
print({r.names[int(i)]: (round(float(a), 3), int(r.nt_per_class[int(i)])) for i, a in zip(r.box.ap_class_index, r.box.ap50)})

Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ds_dyn/data.yaml, degrees=0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=baseline_dyn_raw, nbs=64, nms=None, opset=None, optimize=False, optimizer=aut

In [ ]:
# [步驟5] 掛載 Google Drive，把本次訓練結果複製到 capstone_yolo/run02_v2dynamic2class_baseline/
from google.colab import drive; drive.mount('/content/drive')
import shutil, os
out = '/content/drive/MyDrive/capstone_yolo/run02_v2dynamic2class_baseline'
os.makedirs(out, exist_ok=True)
shutil.copytree('/content/runs/detect/runs/baseline_dyn_raw', out + '/baseline_dyn_raw', dirs_exist_ok=True)
print(os.listdir(out))

Mounted at /content/drive
['baseline_dyn_raw']


# 本版結論（run02）
- **結果**：test mAP50 = 0.688、mAP50-95 = 0.367、P = 0.719、R = 0.679；person AP50 = 0.866（103 個實例）、dynamic_obstacle AP50 = 0.51（31 個實例）。訓練在第 50 epoch 提前停止，最佳為第 30 epoch。
- **與 run01 的比較**：整體 mAP50 由 0.382 升到 0.688，但兩版類別定義不同（run01 為 4 個有樣本的類別平均、run02 為 2 類平均），不能直接視為模型變好。較可比的是 person：0.897 -> 0.866，僅單一 seed、差距 0.03，無法判斷是退步還是訓練波動。合併後的 dynamic_obstacle（0.51）優於 run01 的 motorcycle（0.051）與 bus（0.0），但低於 car（0.58）。
- **現況判斷**：dynamic_obstacle 仍是瓶頸（P 0.657、R 0.495）。train 中 dynamic_obstacle 158 個、person 508 個，比例約 1:3.2；test 的 dynamic_obstacle 只有 31 個，指標變動大，不宜過度解讀。
- **結論**：run02 可作為現有資料下的正式 raw baseline（無擴增、2 類），後續實驗以此為對照；動態障礙物表現不足，支持「優先補資料」的方向，不能靠擴增補足。
- **限制**：僅 seed = 0 一次；未做多 seed，無法估計變異；valid 只有 15 個 dynamic_obstacle，不適合用來挑模型。
- **下一步**：(1) 補拍 dynamic_obstacle（bus / motorcycle / bicycle / car）資料並凍結 test；(2) run03 起在現有資料上做擴增實驗（只擴增 train，一次改一個因素），與本版比較；(3) 以 3 個 seed 重跑 run02 估計變異，再判斷擴增是否有效。
- **輸出位置**：Google Drive/capstone_yolo/run02_v2dynamic2class_baseline/baseline_dyn_raw（已確認複製完成）。